In [1]:
import numpy as np
import torch

def max_pool2d_forward(input_data, kernel_size=2, stride=1, padding=0):
    """
    手动实现二维最大池化前向传播函数
    
    参数:
        input_data: 输入数据，可以是numpy数组或torch张量
                    形状: (batch_size, channels, height, width)
        kernel_size: 池化核大小，整数或二元组(kh, kw)，默认2
        stride: 步幅，整数或二元组(sh, sw)，默认1
        padding: 填充大小，整数或二元组(ph, pw)，默认0
    
    返回:
        output: 池化后的输出，与输入类型相同
                形状: (batch_size, channels, out_height, out_width)
    """
    
    is_tensor = False
    if isinstance(input_data, torch.Tensor):
        is_tensor = True
        input_np = input_data.detach().cpu().numpy()
    else:
        input_np = np.array(input_data)
    
    batch_size, channels, height, width = input_np.shape
    
    if isinstance(kernel_size, int):
        kh, kw = kernel_size, kernel_size
    else:
        kh, kw = kernel_size
    
    if isinstance(stride, int):
        sh, sw = stride, stride
    else:
        sh, sw = stride
    
    if isinstance(padding, int):
        ph, pw = padding, padding
    else:
        ph, pw = padding
    
    input_padded = np.pad(input_np, 
                          ((0, 0), (0, 0), (ph, ph), (pw, pw)),
                          mode='constant',
                          constant_values=-np.inf)
    
    out_height = (height + 2 * ph - kh) // sh + 1
    out_width = (width + 2 * pw - kw) // sw + 1
    
    output_np = np.zeros((batch_size, channels, out_height, out_width))
    
    for b in range(batch_size):
        for c in range(channels):
            for oh in range(out_height):
                for ow in range(out_width):
                    h_start = oh * sh
                    h_end = h_start + kh
                    w_start = ow * sw
                    w_end = w_start + kw
                    
                    window = input_padded[b, c, h_start:h_end, w_start:w_end]
                    output_np[b, c, oh, ow] = np.max(window)
    
    if is_tensor:
        output = torch.tensor(output_np, device=input_data.device, dtype=input_data.dtype)
    else:
        output = output_np
    
    return output

def test_max_pool2d():
    """测试最大池化函数"""
    print("=" * 60)
    print("测试二维最大池化前向传播")
    print("=" * 60)
    
    np.random.seed(42)
    
    print("\n[1] 测试基本功能 (kernel=2, stride=2, padding=0)")
    print("-" * 40)
    
    input_np = np.random.randint(0, 10, (1, 1, 4, 4)).astype(np.float32)
    print("输入张量 (4x4):")
    print(input_np[0, 0])
    
    output_np = max_pool2d_forward(input_np, kernel_size=2, stride=2, padding=0)
    print("\n输出张量 (2x2):")
    print(output_np[0, 0])
    
    print("\n[2] 测试不同步幅 (kernel=2, stride=1, padding=0)")
    print("-" * 40)
    
    output_np = max_pool2d_forward(input_np, kernel_size=2, stride=1, padding=0)
    print("输入张量 (4x4):")
    print(input_np[0, 0])
    print("\n输出张量 (3x3):")
    print(output_np[0, 0])
    
    print("\n[3] 测试填充 (kernel=2, stride=1, padding=1)")
    print("-" * 40)
    
    output_np = max_pool2d_forward(input_np, kernel_size=2, stride=1, padding=1)
    print("输入张量 (4x4):")
    print(input_np[0, 0])
    print("\n输出张量 (5x5):")
    print(output_np[0, 0])
    
    print("\n[4] 测试多通道 (batch=2, channels=3)")
    print("-" * 40)
    
    input_multi = np.random.randint(0, 20, (2, 3, 6, 6)).astype(np.float32)
    output_multi = max_pool2d_forward(input_multi, kernel_size=3, stride=2, padding=1)
    print(f"输入形状: {input_multi.shape}")
    print(f"输出形状: {output_multi.shape}")
    
    print("\n[5] 测试PyTorch张量")
    print("-" * 40)
    
    input_tensor = torch.randint(0, 10, (1, 2, 5, 5)).float()
    print("输入张量:")
    print(input_tensor[0, 0])
    
    output_tensor = max_pool2d_forward(input_tensor, kernel_size=2, stride=2, padding=0)
    print("\n输出张量:")
    print(output_tensor[0, 0])
    
    print("\n[6] 与PyTorch内置MaxPool2d对比验证")
    print("-" * 40)
    
    input_compare = torch.randn(2, 3, 8, 8)
    
    output_custom = max_pool2d_forward(input_compare, kernel_size=3, stride=2, padding=1)
    
    torch_pool = torch.nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    output_torch = torch_pool(input_compare)
    
    print(f"自定义实现输出形状: {output_custom.shape}")
    print(f"PyTorch内置输出形状: {output_torch.shape}")
    
    max_diff = torch.max(torch.abs(output_custom - output_torch))
    print(f"最大差异: {max_diff.item()}")
    
    if max_diff < 1e-6:
        print("验证通过！自定义实现与PyTorch内置结果一致。")
    else:
        print("验证失败！自定义实现与PyTorch内置结果存在差异。")
    
    print("\n[7] 测试非正方形核")
    print("-" * 40)
    
    input_rect = np.random.randint(0, 10, (1, 1, 5, 7)).astype(np.float32)
    print("输入张量 (5x7):")
    print(input_rect[0, 0])
    
    output_rect = max_pool2d_forward(input_rect, kernel_size=(2, 3), stride=(2, 2), padding=(1, 1))
    print("\n输出张量 (kernel=(2,3), stride=(2,2), padding=(1,1)):")
    print(output_rect[0, 0])
    print(f"输出形状: {output_rect.shape}")
    
    print("\n最大池化前向传播测试完成！")

if __name__ == '__main__':
    test_max_pool2d()

测试二维最大池化前向传播

[1] 测试基本功能 (kernel=2, stride=2, padding=0)
----------------------------------------
输入张量 (4x4):
[[6. 3. 7. 4.]
 [6. 9. 2. 6.]
 [7. 4. 3. 7.]
 [7. 2. 5. 4.]]

输出张量 (2x2):
[[9. 7.]
 [7. 7.]]

[2] 测试不同步幅 (kernel=2, stride=1, padding=0)
----------------------------------------
输入张量 (4x4):
[[6. 3. 7. 4.]
 [6. 9. 2. 6.]
 [7. 4. 3. 7.]
 [7. 2. 5. 4.]]

输出张量 (3x3):
[[9. 9. 7.]
 [9. 9. 7.]
 [7. 5. 7.]]

[3] 测试填充 (kernel=2, stride=1, padding=1)
----------------------------------------
输入张量 (4x4):
[[6. 3. 7. 4.]
 [6. 9. 2. 6.]
 [7. 4. 3. 7.]
 [7. 2. 5. 4.]]

输出张量 (5x5):
[[6. 6. 7. 7. 4.]
 [6. 9. 9. 7. 6.]
 [7. 9. 9. 7. 7.]
 [7. 7. 5. 7. 7.]
 [7. 7. 5. 5. 4.]]

[4] 测试多通道 (batch=2, channels=3)
----------------------------------------
输入形状: (2, 3, 6, 6)
输出形状: (2, 3, 3, 3)

[5] 测试PyTorch张量
----------------------------------------
输入张量:
tensor([[3., 4., 6., 2., 6.],
        [6., 8., 9., 4., 6.],
        [3., 7., 1., 3., 2.],
        [2., 9., 2., 9., 4.],
        [7., 6., 8., 1., 7.]])

输

In [2]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    NiN (Network in Network) 块的实现
    
    NiN块的核心结构:
    1. 一个普通卷积层（可指定kernel_size, stride, padding）
    2. 两个1x1卷积层
    3. 每层卷积后紧跟ReLU激活函数
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        kernel_size: 第一个卷积层的核大小（默认为3）
        stride: 第一个卷积层的步幅（默认为1）
        padding: 第一个卷积层的填充（默认为1）
    """
    
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(NiNBlock, self).__init__()
        
        self.block = nn.Sequential(
            # 第一个卷积层：普通卷积
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, 
                      stride=stride, padding=padding),
            nn.ReLU(inplace=True),
            
            # 第二个卷积层：1x1卷积，通道数不变
            nn.Conv2d(out_channels, out_channels, kernel_size=1, stride=1, padding=0),
            nn.ReLU(inplace=True),
            
            # 第三个卷积层：1x1卷积，通道数不变
            nn.Conv2d(out_channels, out_channels, kernel_size=1, stride=1, padding=0),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.block(x)

def test_nin_block():
    """测试NiN块"""
    print("=" * 60)
    print("NiN块 (Network in Network Block) 测试")
    print("=" * 60)
    
    print("\n[1] 创建NiN块实例")
    print("-" * 40)
    
    nin_block = NiNBlock(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
    print(f"NiN块结构:\n{nin_block}")
    
    print("\n[2] 测试前向传播")
    print("-" * 40)
    
    input_tensor = torch.randn(2, 3, 32, 32)
    print(f"输入形状: {input_tensor.shape}")
    
    output_tensor = nin_block(input_tensor)
    print(f"输出形状: {output_tensor.shape}")
    
    print("\n[3] 测试不同参数配置")
    print("-" * 40)
    
    # 测试不同的kernel_size和stride
    nin_block2 = NiNBlock(in_channels=16, out_channels=32, kernel_size=5, stride=2, padding=2)
    output2 = nin_block2(output_tensor)
    print(f"输入形状: {output_tensor.shape}")
    print(f"输出形状 (kernel=5, stride=2): {output2.shape}")
    
    # 测试1x1卷积作为NiN块
    nin_block3 = NiNBlock(in_channels=32, out_channels=64, kernel_size=1, stride=1, padding=0)
    output3 = nin_block3(output2)
    print(f"输入形状: {output2.shape}")
    print(f"输出形状 (kernel=1, stride=1): {output3.shape}")
    
    print("\n[4] 参数量分析")
    print("-" * 40)
    
    def count_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"NiN块(3->16) 参数数量: {count_parameters(nin_block):,}")
    print(f"NiN块(16->32) 参数数量: {count_parameters(nin_block2):,}")
    print(f"NiN块(32->64) 参数数量: {count_parameters(nin_block3):,}")
    
    print("\n[5] 对比传统全连接层参数量")
    print("-" * 40)
    
    # 假设特征图大小为32x32，通道数16
    fc_layer = nn.Linear(16 * 32 * 32, 10)
    print(f"传统全连接层(16*32*32 -> 10) 参数数量: {count_parameters(fc_layer):,}")
    
    # 使用1x1卷积实现通道变换
    conv_1x1 = nn.Conv2d(16, 10, kernel_size=1)
    print(f"1x1卷积层(16 -> 10) 参数数量: {count_parameters(conv_1x1):,}")
    
    print("\n[6] 构建完整的NiN网络示例")
    print("-" * 40)
    
    class NiN(nn.Module):
        def __init__(self, num_classes=10):
            super(NiN, self).__init__()
            self.features = nn.Sequential(
                NiNBlock(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=0),
                nn.MaxPool2d(kernel_size=3, stride=2),
                
                NiNBlock(in_channels=96, out_channels=256, kernel_size=5, stride=1, padding=2),
                nn.MaxPool2d(kernel_size=3, stride=2),
                
                NiNBlock(in_channels=256, out_channels=384, kernel_size=3, stride=1, padding=1),
                nn.MaxPool2d(kernel_size=3, stride=2),
                
                nn.Dropout(0.5),
                NiNBlock(in_channels=384, out_channels=num_classes, kernel_size=3, stride=1, padding=1),
                nn.AdaptiveAvgPool2d((1, 1)),
                nn.Flatten()
            )
        
        def forward(self, x):
            return self.features(x)
    
    nin_model = NiN(num_classes=10)
    print(f"完整NiN网络结构:\n{nin_model}")
    
    test_input = torch.randn(2, 3, 224, 224)
    test_output = nin_model(test_input)
    print(f"\nNiN网络输入形状: {test_input.shape}")
    print(f"NiN网络输出形状: {test_output.shape}")
    print(f"NiN网络总参数数量: {count_parameters(nin_model):,}")
    
    print("\nNiN块测试完成！")

if __name__ == '__main__':
    test_nin_block()

NiN块 (Network in Network Block) 测试

[1] 创建NiN块实例
----------------------------------------
NiN块结构:
NiNBlock(
  (block): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU(inplace=True)
    (4): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU(inplace=True)
  )
)

[2] 测试前向传播
----------------------------------------
输入形状: torch.Size([2, 3, 32, 32])
输出形状: torch.Size([2, 16, 32, 32])

[3] 测试不同参数配置
----------------------------------------
输入形状: torch.Size([2, 16, 32, 32])
输出形状 (kernel=5, stride=2): torch.Size([2, 32, 16, 16])
输入形状: torch.Size([2, 32, 16, 16])
输出形状 (kernel=1, stride=1): torch.Size([2, 64, 16, 16])

[4] 参数量分析
----------------------------------------
NiN块(3->16) 参数数量: 992
NiN块(16->32) 参数数量: 14,944
NiN块(32->64) 参数数量: 10,432

[5] 对比传统全连接层参数量
----------------------------------------
传统全连接层(16*32*32 -> 10) 参数数量: 163,850
1x1卷积层(16 -> 

In [3]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    ResNet残差块实现
    
    结构:
    - 第一个3x3卷积层 -> BatchNorm -> ReLU
    - 第二个3x3卷积层 -> BatchNorm
    - 残差连接: 如果use_1x1conv=True，对输入进行1x1卷积调整
    - 最终输出: relu(卷积输出 + 残差连接)
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        use_1x1conv: 是否使用1x1卷积调整输入形状
        stride: 第一个卷积层的步幅（默认为1）
    """
    
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        
        # 第一个卷积层: 3x3
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        # 第二个卷积层: 3x3
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                               stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 1x1卷积用于调整输入通道数和形状
        if use_1x1conv:
            self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                                     stride=stride)
        else:
            self.conv1x1 = None
    
    def forward(self, x):
        # 主路径
        y = self.conv1(x)
        y = self.bn1(y)
        y = self.relu(y)
        
        y = self.conv2(y)
        y = self.bn2(y)
        
        # 残差路径
        if self.conv1x1 is not None:
            x = self.conv1x1(x)
        
        # 残差连接: f(x) + x
        y += x
        y = self.relu(y)
        
        return y

def test_residual_block():
    """测试残差块"""
    print("=" * 60)
    print("ResNet残差块 (Residual Block) 测试")
    print("=" * 60)
    
    print("\n[1] 创建残差块实例 (不使用1x1卷积)")
    print("-" * 40)
    
    res_block1 = Residual(in_channels=64, out_channels=64)
    print(f"残差块结构:\n{res_block1}")
    
    print("\n[2] 创建残差块实例 (使用1x1卷积调整通道)")
    print("-" * 40)
    
    res_block2 = Residual(in_channels=64, out_channels=128, use_1x1conv=True, stride=2)
    print(f"残差块结构:\n{res_block2}")
    
    print("\n[3] 测试前向传播 (通道数不变)")
    print("-" * 40)
    
    input_tensor = torch.randn(2, 64, 32, 32)
    print(f"输入形状: {input_tensor.shape}")
    
    output_tensor = res_block1(input_tensor)
    print(f"输出形状: {output_tensor.shape}")
    print(f"通道数保持不变: {input_tensor.shape[1] == output_tensor.shape[1]}")
    
    print("\n[4] 测试前向传播 (通道数变化)")
    print("-" * 40)
    
    input_tensor2 = torch.randn(2, 64, 32, 32)
    print(f"输入形状: {input_tensor2.shape}")
    
    output_tensor2 = res_block2(input_tensor2)
    print(f"输出形状: {output_tensor2.shape}")
    print(f"通道数从64增加到128: {output_tensor2.shape[1] == 128}")
    print(f"特征图尺寸减半: {output_tensor2.shape[2] == 16}")
    
    print("\n[5] 测试多个残差块堆叠")
    print("-" * 40)
    
    # 模拟ResNet的一个阶段
    block1 = Residual(128, 128)
    block2 = Residual(128, 128)
    block3 = Residual(128, 256, use_1x1conv=True, stride=2)
    
    input_stage = torch.randn(2, 128, 16, 16)
    x = block1(input_stage)
    x = block2(x)
    x = block3(x)
    
    print(f"阶段输入形状: {input_stage.shape}")
    print(f"阶段输出形状: {x.shape}")
    
    print("\n[6] 参数量分析")
    print("-" * 40)
    
    def count_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"残差块(64->64, 无1x1) 参数数量: {count_parameters(res_block1):,}")
    print(f"残差块(64->128, 有1x1) 参数数量: {count_parameters(res_block2):,}")
    
    print("\n[7] 验证残差连接的梯度流动")
    print("-" * 40)
    
    res_block = Residual(32, 32)
    input_grad = torch.randn(1, 32, 8, 8, requires_grad=True)
    output_grad = res_block(input_grad)
    
    # 计算梯度
    loss = output_grad.sum()
    loss.backward()
    
    print(f"输入梯度形状: {input_grad.grad.shape}")
    print(f"梯度范数: {torch.norm(input_grad.grad):.4f}")
    print("梯度成功回传，验证了残差连接的有效性")
    
    print("\n[8] 构建ResNet-18示例")
    print("-" * 40)
    
    def resnet_block(in_channels, out_channels, num_residuals, first_block=False):
        blk = []
        for i in range(num_residuals):
            if i == 0 and not first_block:
                blk.append(Residual(in_channels, out_channels, use_1x1conv=True, stride=2))
            else:
                blk.append(Residual(out_channels, out_channels))
        return nn.Sequential(*blk)
    
    class ResNet18(nn.Module):
        def __init__(self, num_classes=10):
            super(ResNet18, self).__init__()
            self.net = nn.Sequential(
                nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
                
                resnet_block(64, 64, 2, first_block=True),
                resnet_block(64, 128, 2),
                resnet_block(128, 256, 2),
                resnet_block(256, 512, 2),
                
                nn.AdaptiveAvgPool2d((1, 1)),
                nn.Flatten(),
                nn.Linear(512, num_classes)
            )
        
        def forward(self, x):
            return self.net(x)
    
    resnet18 = ResNet18()
    print(f"ResNet-18总参数数量: {count_parameters(resnet18):,}")
    
    test_input = torch.randn(2, 3, 224, 224)
    test_output = resnet18(test_input)
    print(f"ResNet-18输入形状: {test_input.shape}")
    print(f"ResNet-18输出形状: {test_output.shape}")
    
    print("\n残差块测试完成！")

if __name__ == '__main__':
    test_residual_block()

ResNet残差块 (Residual Block) 测试

[1] 创建残差块实例 (不使用1x1卷积)
----------------------------------------
残差块结构:
Residual(
  (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)

[2] 创建残差块实例 (使用1x1卷积调整通道)
----------------------------------------
残差块结构:
Residual(
  (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv1x1): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2))
)

[3] 测试前

In [4]:
import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

def create_augmentation_pipeline():
    """
    创建图像增广管道
    
    包含以下操作:
    1. 随机裁剪（面积比例0.08-1.0）并缩放到224×224
    2. 50%概率水平翻转
    3. 随机改变亮度、对比度、饱和度（变化范围0.5）
    4. 转换为PyTorch张量
    
    返回:
        transforms.Compose: 组合的图像增广管道
    """
    pipeline = transforms.Compose([
        # 随机裁剪并缩放
        transforms.RandomResizedCrop(
            size=(224, 224),
            scale=(0.08, 1.0),
            ratio=(3/4, 4/3)  # 宽高比范围
        ),
        # 50%概率水平翻转
        transforms.RandomHorizontalFlip(p=0.5),
        # 随机颜色抖动（亮度、对比度、饱和度）
        transforms.ColorJitter(
            brightness=0.5,
            contrast=0.5,
            saturation=0.5,
            hue=0.1  # 可选：添加微小的色相变化
        ),
        # 转换为张量
        transforms.ToTensor()
    ])
    
    return pipeline

def test_augmentation_pipeline():
    """测试图像增广管道"""
    print("=" * 60)
    print("图像增广管道测试")
    print("=" * 60)
    
    # 创建增广管道
    aug_pipeline = create_augmentation_pipeline()
    print(f"增广管道:\n{aug_pipeline}")
    
    print("\n[1] 加载测试图像")
    print("-" * 40)
    
    # 创建一个简单的测试图像（3通道，256x256）
    test_img = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
    pil_img = Image.fromarray(test_img)
    print(f"原始图像形状: {test_img.shape}")
    
    print("\n[2] 测试增广管道")
    print("-" * 40)
    
    # 多次应用增广，展示不同的效果
    num_augmentations = 5
    augmented_images = []
    
    for i in range(num_augmentations):
        augmented = aug_pipeline(pil_img)
        augmented_images.append(augmented)
        print(f"增广图像{i+1}形状: {augmented.shape}, 数据类型: {augmented.dtype}")
    
    print("\n[3] 验证张量属性")
    print("-" * 40)
    
    sample_tensor = augmented_images[0]
    print(f"张量形状: {sample_tensor.shape} (C, H, W)")
    print(f"张量最小值: {sample_tensor.min().item():.4f}")
    print(f"张量最大值: {sample_tensor.max().item():.4f}")
    print(f"张量均值: {sample_tensor.mean().item():.4f}")
    
    print("\n[4] 测试不同的增广参数")
    print("-" * 40)
    
    # 测试只包含部分增广的管道
    pipeline_no_flip = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
        transforms.ToTensor()
    ])
    
    pipeline_no_color = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor()
    ])
    
    img_no_flip = pipeline_no_flip(pil_img)
    img_no_color = pipeline_no_color(pil_img)
    
    print(f"无翻转管道输出形状: {img_no_flip.shape}")
    print(f"无颜色抖动管道输出形状: {img_no_color.shape}")
    
    print("\n[5] 测试组合使用")
    print("-" * 40)
    
    # 测试在数据加载中使用
    from torch.utils.data import Dataset, DataLoader
    
    class TestDataset(Dataset):
        def __init__(self, num_samples=10, transform=None):
            self.num_samples = num_samples
            self.transform = transform
        
        def __len__(self):
            return self.num_samples
        
        def __getitem__(self, idx):
            # 生成随机图像
            img = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
            img = Image.fromarray(img)
            
            if self.transform:
                img = self.transform(img)
            
            return img
    
    dataset = TestDataset(num_samples=10, transform=aug_pipeline)
    dataloader = DataLoader(dataset, batch_size=2, shuffle=True)
    
    for batch in dataloader:
        print(f"批次形状: {batch.shape}")
        break
    
    print("\n[6] 增广管道组件说明")
    print("-" * 40)
    print("RandomResizedCrop: 随机裁剪并保持宽高比，缩放至指定尺寸")
    print("RandomHorizontalFlip: 水平翻转，概率p=0.5")
    print("ColorJitter: 随机调整亮度、对比度、饱和度、色相")
    print("ToTensor: 将PIL图像转换为PyTorch张量，像素值归一化到[0,1]")
    
    print("\n图像增广管道测试完成！")

if __name__ == '__main__':
    test_augmentation_pipeline()

图像增广管道测试
增广管道:
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=(-0.1, 0.1))
    ToTensor()
)

[1] 加载测试图像
----------------------------------------
原始图像形状: (256, 256, 3)

[2] 测试增广管道
----------------------------------------
增广图像1形状: torch.Size([3, 224, 224]), 数据类型: torch.float32
增广图像2形状: torch.Size([3, 224, 224]), 数据类型: torch.float32
增广图像3形状: torch.Size([3, 224, 224]), 数据类型: torch.float32
增广图像4形状: torch.Size([3, 224, 224]), 数据类型: torch.float32
增广图像5形状: torch.Size([3, 224, 224]), 数据类型: torch.float32

[3] 验证张量属性
----------------------------------------
张量形状: torch.Size([3, 224, 224]) (C, H, W)
张量最小值: 0.0000
张量最大值: 1.0000
张量均值: 0.4825

[4] 测试不同的增广参数
----------------------------------------
无翻转管道输出形状: torch.Size([3, 224, 224])
无颜色抖动管道输出形状: torch.Size([3, 224, 224])

[5] 测试组合使用
-------------------

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LabelSmoothingCrossEntropy(nn.Module):
    """
    标签平滑交叉熵损失函数
    
    原理:
    - 标准交叉熵使用独热编码 (One-hot)
    - 设置平滑因子 epsilon，对于K分类问题：
      - 真实标签目标概率: 1 - epsilon
      - 其他错误类别目标概率: epsilon / (K - 1)
    
    例如 epsilon=0.1, K=3:
      类别0: [1, 0, 0] -> [0.9, 0.05, 0.05]
      类别1: [0, 1, 0] -> [0.05, 0.9, 0.05]
      类别2: [0, 0, 1] -> [0.05, 0.05, 0.9]
    
    参数:
        epsilon: 平滑因子，默认0.1
    """
    
    def __init__(self, epsilon=0.1):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.epsilon = epsilon
    
    def forward(self, logits, targets):
        """
        计算标签平滑交叉熵损失
        
        参数:
            logits: 模型输出的原始分数，形状 (batch_size, num_classes)
            targets: 真实类别标签，形状 (batch_size,) 或独热编码
        
        返回:
            loss: 平均损失值
        """
        K = logits.size(1)
        
        if targets.dim() == 1:
            # 如果targets是类别索引，转换为独热编码
            targets_one_hot = F.one_hot(targets, K).float()
        else:
            targets_one_hot = targets.float()
        
        # 应用标签平滑
        smoothed_targets = (1 - self.epsilon) * targets_one_hot + \
                         self.epsilon / (K - 1) * (1 - targets_one_hot)
        
        # 计算交叉熵损失
        log_probs = F.log_softmax(logits, dim=1)
        loss = -torch.sum(smoothed_targets * log_probs, dim=1)
        
        return loss.mean()

class LabelSmoothingCrossEntropyV2(nn.Module):
    """
    标签平滑交叉熵损失的优化实现
    使用数学等价形式避免显式构造独热编码，提高数值稳定性
    """
    
    def __init__(self, epsilon=0.1):
        super(LabelSmoothingCrossEntropyV2, self).__init__()
        self.epsilon = epsilon
    
    def forward(self, logits, targets):
        """
        优化版本的标签平滑交叉熵损失
        
        数学推导:
        loss = -∑ q_i * log(p_i)
              = -q_target * log(p_target) - ∑_{i≠target} q_i * log(p_i)
              = -(1-ε) * log(p_target) - ε/(K-1) * ∑_{i≠target} log(p_i)
        """
        K = logits.size(1)
        log_probs = F.log_softmax(logits, dim=1)
        
        if targets.dim() == 1:
            # 获取目标类别的对数概率
            target_log_probs = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
            # 所有类别的对数概率之和
            sum_log_probs = log_probs.sum(dim=1)
            # 非目标类别的对数概率之和
            non_target_log_probs = sum_log_probs - target_log_probs
            
            # 应用标签平滑公式
            loss = -(1 - self.epsilon) * target_log_probs - \
                   self.epsilon / (K - 1) * non_target_log_probs
        else:
            # 独热编码形式
            smoothed_targets = (1 - self.epsilon) * targets + \
                             self.epsilon / (K - 1) * (1 - targets)
            loss = -(smoothed_targets * log_probs).sum(dim=1)
        
        return loss.mean()

def test_label_smoothing():
    """测试标签平滑交叉熵损失"""
    print("=" * 60)
    print("标签平滑交叉熵损失 (Label Smoothing Cross Entropy) 测试")
    print("=" * 60)
    
    epsilon = 0.1
    num_classes = 5
    batch_size = 4
    
    print(f"\n[1] 参数设置")
    print("-" * 40)
    print(f"平滑因子 epsilon: {epsilon}")
    print(f"类别数量 K: {num_classes}")
    print(f"批量大小: {batch_size}")
    
    print(f"\n平滑后的标签分布:")
    print(f"  真实类别目标概率: {1 - epsilon:.2f}")
    print(f"  其他类别目标概率: {epsilon / (num_classes - 1):.4f}")
    
    print(f"\n示例 (K={num_classes}, epsilon={epsilon}):")
    for i in range(num_classes):
        smoothed = [0.0] * num_classes
        smoothed[i] = 1 - epsilon
        for j in range(num_classes):
            if j != i:
                smoothed[j] = epsilon / (num_classes - 1)
        print(f"  类别{i}: {[f'{x:.2f}' for x in smoothed]}")
    
    print("\n[2] 创建损失函数实例")
    print("-" * 40)
    
    criterion1 = LabelSmoothingCrossEntropy(epsilon=epsilon)
    criterion2 = LabelSmoothingCrossEntropyV2(epsilon=epsilon)
    criterion_standard = nn.CrossEntropyLoss()
    
    print(f"标准交叉熵损失: {criterion_standard}")
    print(f"标签平滑损失 (v1): {criterion1}")
    print(f"标签平滑损失 (v2): {criterion2}")
    
    print("\n[3] 测试前向传播")
    print("-" * 40)
    
    # 模拟模型输出的logits
    torch.manual_seed(42)
    logits = torch.randn(batch_size, num_classes)
    targets = torch.randint(0, num_classes, (batch_size,))
    
    print(f"模型logits:\n{logits}")
    print(f"真实标签: {targets}")
    
    # 计算各种损失
    loss_standard = criterion_standard(logits, targets)
    loss_smooth_v1 = criterion1(logits, targets)
    loss_smooth_v2 = criterion2(logits, targets)
    
    print(f"\n损失对比:")
    print(f"  标准交叉熵损失: {loss_standard.item():.6f}")
    print(f"  标签平滑损失 (v1): {loss_smooth_v1.item():.6f}")
    print(f"  标签平滑损失 (v2): {loss_smooth_v2.item():.6f}")
    
    print("\n[4] 验证v1和v2实现的一致性")
    print("-" * 40)
    
    max_diff = torch.abs(loss_smooth_v1 - loss_smooth_v2).item()
    print(f"v1和v2的最大差异: {max_diff:.10f}")
    
    if max_diff < 1e-6:
        print("验证通过！两种实现结果一致。")
    
    print("\n[5] 测试不同epsilon值的影响")
    print("-" * 40)
    
    test_epsilons = [0.0, 0.1, 0.2, 0.5]
    for eps in test_epsilons:
        criterion = LabelSmoothingCrossEntropy(epsilon=eps)
        loss = criterion(logits, targets)
        print(f"epsilon={eps:.1f}: 损失={loss.item():.6f}")
    
    print("\n[6] 测试独热编码标签输入")
    print("-" * 40)
    
    targets_one_hot = F.one_hot(targets, num_classes).float()
    loss_one_hot = criterion1(logits, targets_one_hot)
    print(f"独热编码标签损失: {loss_one_hot.item():.6f}")
    
    print("\n[7] 标签平滑的效果分析")
    print("-" * 40)
    
    print("\n标签平滑的优势:")
    print("1. 防止模型对训练数据过于自信")
    print("2. 提高模型在未见过的数据上的泛化能力")
    print("3. 减少过拟合的风险")
    print("4. 使损失函数更加平滑，优化过程更稳定")
    
    print("\n标签平滑的数学原理:")
    print("- 标准交叉熵: 鼓励预测概率分布接近one-hot")
    print("- 标签平滑: 允许一定的概率分配给非目标类别")
    print("- 等效于在损失函数中加入了正则化项")
    
    print("\n标签平滑交叉熵损失测试完成！")

if __name__ == '__main__':
    test_label_smoothing()

标签平滑交叉熵损失 (Label Smoothing Cross Entropy) 测试

[1] 参数设置
----------------------------------------
平滑因子 epsilon: 0.1
类别数量 K: 5
批量大小: 4

平滑后的标签分布:
  真实类别目标概率: 0.90
  其他类别目标概率: 0.0250

示例 (K=5, epsilon=0.1):
  类别0: ['0.90', '0.03', '0.03', '0.03', '0.03']
  类别1: ['0.03', '0.90', '0.03', '0.03', '0.03']
  类别2: ['0.03', '0.03', '0.90', '0.03', '0.03']
  类别3: ['0.03', '0.03', '0.03', '0.90', '0.03']
  类别4: ['0.03', '0.03', '0.03', '0.03', '0.90']

[2] 创建损失函数实例
----------------------------------------
标准交叉熵损失: CrossEntropyLoss()
标签平滑损失 (v1): LabelSmoothingCrossEntropy()
标签平滑损失 (v2): LabelSmoothingCrossEntropyV2()

[3] 测试前向传播
----------------------------------------
模型logits:
tensor([[ 1.9269,  1.4873,  0.9007, -2.1055, -0.7581],
        [ 1.0783,  0.8008,  1.6806,  0.3559, -0.6866],
        [-0.4934,  0.2415, -0.2316,  0.0418, -0.2516],
        [ 0.8599, -0.3097, -0.3957,  0.8034, -0.6216]])
真实标签: tensor([4, 1, 2, 0])

损失对比:
  标准交叉熵损失: 1.974232
  标签平滑损失 (v1): 1.968178
  标签平滑损失 (v2): 1.968178

[